# **CNN Cat & Dog Image Classifiation**

- This notebook implements a binary image classifier to distinguish between cats and dogs using transfer learning in TensorFlow/Keras.

- users upload a custom image to test live cat-or-dog predictions with confidence scores.

- It enhances a baseline model to reduce overfitting through data augmentation, an ImageNet-pretrained MobileNetV2 backbone, early stopping callbacks, and fine-tuning, before exporting the final model to TensorFlow Lite format for deployment.

## **Setup & Download Dataset**

Downloads the raw cat & dog image dataset from Kaggle so it's available locally in this environment.


In [ ]:
# Download the 'cat-and-dog' dataset from Kaggle and unzip it into /content
!kaggle datasets download -d tongpython/cat-and-dog -p /content --unzip


Dataset URL: https://www.kaggle.com/datasets/tongpython/cat-and-dog
License(s): CC0-1.0
100% 218M/218M [00:02<00:00, 99.8MB/s]



In [ ]:
from google.colab import drive
drive.mount('/content/drive')  # mount Google Drive so Colab can read files from it

# Update these paths to match your dataset location
TRAIN_DIR = '/content/drive/MyDrive/Dog_Cat_CNN/training_set'  # folder containing training images
TEST_DIR = '/content/drive/MyDrive/Dog_Cat_CNN/test_set'  # folder containing test/validation images


Mounted at /content/drive


## **Import Libraries**

Loads TensorFlow/Keras utilities for building the model, plus NumPy and Matplotlib for data handling and plotting.


In [ ]:
import tensorflow as tf  # core deep learning framework
from tensorflow.keras.preprocessing import image_dataset_from_directory  # loads images from folders into a tf.data.Dataset
from tensorflow.keras.applications import MobileNetV2  # pretrained CNN backbone used for transfer learning
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input  # normalizes images the way MobileNetV2 expects
from tensorflow.keras import layers, models, callbacks  # building blocks for defining and training the model
import matplotlib.pyplot as plt  # for plotting accuracy/loss curves
import numpy as np  # numerical operations on arrays

## **Load Datasets**

Reads the training and validation images from disk into batched, labeled `tf.data.Dataset` objects the model can train on.


In [ ]:
BATCH_SIZE = 16  # number of images processed together in one training step
IMG_SIZE = (150, 150)  # all images are resized to this size

train_dataset = image_dataset_from_directory(
    TRAIN_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='binary',  # labels are 0/1 (cat/dog) since this is binary classification
    shuffle=True,  # shuffle order each epoch so the model doesn't learn a fixed sequence
    seed=42  # fixed seed so shuffling is reproducible
)

validation_dataset = image_dataset_from_directory(
    TEST_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='binary'
)

class_names = train_dataset.class_names  # e.g. ['cats', 'dogs'], inferred from subfolder names
print("Classes:", class_names)

# Prefetch for performance
AUTOTUNE = tf.data.AUTOTUNE  # lets TensorFlow decide the optimal number of batches to prefetch
train_dataset = train_dataset.prefetch(buffer_size=AUTOTUNE)  # overlaps data loading with model training
validation_dataset = validation_dataset.prefetch(buffer_size=AUTOTUNE)

Found 7896 files belonging to 2 classes.
Found 1997 files belonging to 2 classes.
Classes: ['cats', 'dogs']


## **Data Augmentation**

Randomly flips, rotates, zooms, and adjusts contrast on training images each epoch so the model sees more variation and stops memorizing exact training samples (this fights overfitting).


In [ ]:
data_augmentation = models.Sequential([
    layers.RandomFlip("horizontal"),  # randomly mirrors images left-right
    layers.RandomRotation(0.15),  # randomly rotates images by up to ~15% of a full turn
    layers.RandomZoom(0.15),  # randomly zooms in/out by up to 15%
    layers.RandomContrast(0.1),  # randomly tweaks image contrast
], name="data_augmentation")

## **Build the Model - (Transfer Learning with MobileNetV2)**

Instead of training a CNN from scratch on 8,000 images (which overfits fast), we reuse a MobileNetV2 backbone pretrained on ImageNet and only train a small classification head on top.

Note: MobileNetV2 expects its own **preprocess_input** normalization rather than simple **255** rescaling.


In [ ]:
base_model = MobileNetV2(
    input_shape=(150, 150, 3),
    include_top=False,  # exclude MobileNetV2's original classification layer, we'll add our own
    weights='imagenet'  # load weights pretrained on the ImageNet dataset
)
base_model.trainable = False  # freeze base for initial training so pretrained features aren't overwritten

inputs = tf.keras.Input(shape=(150, 150, 3))  # defines the input shape of the model
x = data_augmentation(inputs)  # apply random augmentations (only active during training)
x = preprocess_input(x)  # scale pixel values the way MobileNetV2 was trained to expect
x = base_model(x, training=False)  # run through the frozen pretrained backbone
x = layers.GlobalAveragePooling2D()(x)  # collapse spatial feature maps into a single feature vector
x = layers.Dropout(0.3)(x)  # randomly drop 30% of activations during training to reduce overfitting
outputs = layers.Dense(1, activation='sigmoid')(x)  # single output neuron, probability of "dog"

model = models.Model(inputs, outputs)  # wire inputs and outputs into a trainable model

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),  # Adam optimizer with a moderate learning rate
    loss='binary_crossentropy',  # standard loss for binary (cat/dog) classification
    metrics=['accuracy']
)

model.summary()  # print the model architecture and parameter counts

/tmp/ipykernel_933/4224769611.py:1: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = MobileNetV2(


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 150, 150, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ data_augmentation (Sequential)  │ (None, 150, 150, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ true_divide (TrueDivide)        │ (None, 150, 150, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ subtract (Subtract)             │ (None, 150, 150, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 5, 5, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │         1,281 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,259,265 (8.62 MB)

 Trainable params: 1,281 (5.00 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

## **Callbacks**

Utilities that monitor training and automatically stop early or save the best-performing model, so we don't overtrain or lose good weights.


In [ ]:
early_stop = callbacks.EarlyStopping(
    monitor='val_loss',  # watch validation loss
    patience=3,  # stop if it doesn't improve for 3 consecutive epochs
    restore_best_weights=True  # roll back to the best epoch's weights when stopping
)

checkpoint = callbacks.ModelCheckpoint(
    'best_cat_dog_model.keras',  # file to save the best model to
    monitor='val_loss',
    save_best_only=True  # only overwrite the saved file when validation loss improves
)

## **Train (Phase 1 - Freeze Base)**

Trains only the new classification head while the MobileNetV2 backbone stays frozen, so the pretrained features aren't disturbed yet.


In [ ]:
EPOCHS = 20  # maximum number of passes over the training data (EarlyStopping may stop sooner)

history = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=EPOCHS,
    callbacks=[early_stop, checkpoint]  # apply early stopping and checkpointing during training
)

Epoch 1/20
494/494 ━━━━━━━━━━━━━━━━━━━━ 916s 2s/step - accuracy: 0.8950 - loss: 0.2482 - val_accuracy: 0.9715 - val_loss: 0.0795
Epoch 2/20
494/494 ━━━━━━━━━━━━━━━━━━━━ 257s 520ms/step - accuracy: 0.9301 - loss: 0.1729 - val_accuracy: 0.9730 - val_loss: 0.0738
Epoch 3/20
494/494 ━━━━━━━━━━━━━━━━━━━━ 250s 496ms/step - accuracy: 0.9311 - loss: 0.1736 - val_accuracy: 0.9780 - val_loss: 0.0715
Epoch 4/20
494/494 ━━━━━━━━━━━━━━━━━━━━ 255s 482ms/step - accuracy: 0.9377 - loss: 0.1590 - val_accuracy: 0.9790 - val_loss: 0.0688
Epoch 5/20
494/494 ━━━━━━━━━━━━━━━━━━━━ 246s 498ms/step - accuracy: 0.9345 - loss: 0.1659 - val_accuracy: 0.9740 - val_loss: 0.0741
Epoch 6/20
494/494 ━━━━━━━━━━━━━━━━━━━━ 286s 578ms/step - accuracy: 0.9401 - loss: 0.1521 - val_accuracy: 0.9785 - val_loss: 0.0708
Epoch 7/20
494/494 ━━━━━━━━━━━━━━━━━━━━ 242s 490ms/step - accuracy: 0.9352 - loss: 0.1652 - val_accuracy: 0.9800 - val_loss: 0.0694


## **Fine-Tune (Phase 2 - Unfreeze Top Layers)**

Once the head has converged, unfreeze the last layers of MobileNetV2 and continue training with a low learning rate to squeeze out extra accuracy without destroying the pretrained features.


In [ ]:
base_model.trainable = True  # unfreeze the whole backbone so some of its layers can be updated

# Freeze all layers except the last 30
for layer in base_model.layers[:-30]:
    layer.trainable = False  # keep earlier (more general) layers frozen; only later layers adapt

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),  # very small learning rate to avoid wrecking pretrained weights
    loss='binary_crossentropy',
    metrics=['accuracy']
)

fine_tune_epochs = 10  # additional epochs to train during fine-tuning
total_epochs = len(history.epoch) + fine_tune_epochs  # continue epoch counting from where Phase 1 left off

history_fine = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=total_epochs,
    initial_epoch=history.epoch[-1] + 1,  # resume epoch numbering instead of starting back at 0
    callbacks=[early_stop, checkpoint]
)

Epoch 8/17
494/494 ━━━━━━━━━━━━━━━━━━━━ 332s 647ms/step - accuracy: 0.8775 - loss: 0.2751 - val_accuracy: 0.9775 - val_loss: 0.0645
Epoch 9/17
494/494 ━━━━━━━━━━━━━━━━━━━━ 358s 726ms/step - accuracy: 0.9136 - loss: 0.2108 - val_accuracy: 0.9805 - val_loss: 0.0581
Epoch 10/17
494/494 ━━━━━━━━━━━━━━━━━━━━ 323s 654ms/step - accuracy: 0.9219 - loss: 0.1863 - val_accuracy: 0.9785 - val_loss: 0.0549
Epoch 11/17
494/494 ━━━━━━━━━━━━━━━━━━━━ 333s 673ms/step - accuracy: 0.9268 - loss: 0.1795 - val_accuracy: 0.9790 - val_loss: 0.0551
Epoch 12/17
494/494 ━━━━━━━━━━━━━━━━━━━━ 416s 743ms/step - accuracy: 0.9301 - loss: 0.1693 - val_accuracy: 0.9790 - val_loss: 0.0552
Epoch 13/17
494/494 ━━━━━━━━━━━━━━━━━━━━ 324s 656ms/step - accuracy: 0.9328 - loss: 0.1684 - val_accuracy: 0.9800 - val_loss: 0.0548
Epoch 14/17
494/494 ━━━━━━━━━━━━━━━━━━━━ 419s 731ms/step - accuracy: 0.9382 - loss: 0.1525 - val_accuracy: 0.9805 - val_loss: 0.0554
Epoch 15/17
494/494 ━━━━━━━━━━━━━━━━━━━━ 332s 672ms/step - accuracy: 0.

## **Evaluate on Test Set**

Checks final model performance on the training set and the held-out validation/test set to see how well it generalizes.


In [ ]:
# Evaluate the model on the training dataset
train_loss, train_accuracy = model.evaluate(train_dataset)  # compute loss & accuracy on training data

print(f"Training Loss: {train_loss:.4f}")
print(f"Training Accuracy: {train_accuracy * 100:.2f}%")

494/494 ━━━━━━━━━━━━━━━━━━━━ 186s 377ms/step - accuracy: 0.9867 - loss: 0.0420
Training Loss: 0.0420
Training Accuracy: 98.67%


In [ ]:
loss, accuracy = model.evaluate(validation_dataset)  # compute loss & accuracy on validation/test data
print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy * 100:.2f}%")

125/125 ━━━━━━━━━━━━━━━━━━━━ 48s 386ms/step - accuracy: 0.9810 - loss: 0.0534
Test Loss: 0.0534
Test Accuracy: 98.10%


## **Save the Final Model**

Persists the trained model to disk in Keras format so it can be reloaded later without retraining.


In [ ]:
model.save("cat_dog_cnn_model.keras")  # save architecture + weights + optimizer state in one file
print("Model saved as cat_dog_cnn_model.keras")

Model saved as cat_dog_cnn_model.keras


## **Convert to TFLite**

Converts the Keras model into a lightweight TensorFlow Lite (.tflite) format suitable for mobile/edge deployment. Same conversion step as your original **convert.py**, updated to point at the new model.


In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)  # create a converter from the trained Keras model
converter.optimizations = [tf.lite.Optimize.DEFAULT]  # apply default optimizations (e.g. quantization) to shrink model size
tflite_model = converter.convert()  # perform the actual conversion

with open("cat_dog_cnn_model.tflite", "wb") as f:
    f.write(tflite_model)  # write the converted model bytes to disk

print("Saved cat_dog_cnn_model.tflite successfully!")

Saved artifact at '/tmp/tmp5ia1x8ri'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 150, 150, 3), dtype=tf.float32, name='keras_tensor_154')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  139665161459408: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139665161460176: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139665161459984: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139665161460368: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139665161457872: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139665161459600: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139665161461136: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139665161460944: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139665161461328: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139665161459216: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1396651614

## **Predict on a New Image**

Lets you upload a new photo and runs it through the trained model to classify it as cat or dog.


In [ ]:
import numpy as np
from tensorflow.keras.preprocessing import image  # utilities for loading/converting images
from google.colab import files  # Colab widget for uploading files from your computer

uploaded = files.upload()  # opens a file picker; returns a dict of {filename: bytes}

for fn in uploaded.keys():
    img = image.load_img(fn, target_size=(150, 150))  # load and resize the uploaded image
    x = image.img_to_array(img)  # convert the image to a NumPy array
    x = np.expand_dims(x, axis=0)  # add a batch dimension; no manual /255.0 — preprocess_input is baked into the model

    prediction = model.predict(x)[0][0]  # run inference; output is a probability between 0 and 1

    print(f"\nFile: {fn}")
    if prediction > 0.5:
        print(f"Prediction: DOG ({prediction * 100:.2f}% confidence)")  # closer to 1 means "dog"
    else:
        print(f"Prediction: CAT ({(1 - prediction) * 100:.2f}% confidence)")  # closer to 0 means "cat"

Saving download (1).jfif to download (1).jfif
1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step

File: download (1).jfif
Prediction: CAT (93.43% confidence)


## **Conclusion**

1. Transfer learning worked well **MobileNetV2 (pretrained)** avoided overfitting despite the small dataset.

2. **Fine-tuning** improved results unfreezing top layers pushed validation accuracy to **98%**.

3. Strong, generalized performance **98.67% train / 98.10% test accuracy**, close scores showing good efficacy (ability to reliably produce the desired result).

4. Callbacks added robustness early stopping and checkpointing made training more judicious (careful and well-reasoned), preventing wasted epochs and preserving the best-performing weights.

5. Deployment-ready exported to TFLite and successfully tested on a new **image (CAT, 93.43% confidence)**.